In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------
# Metric diagnostics
# -----------------------------
def metric_diagnostics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    y_true_log = np.log1p(y_true)
    y_pred_log = np.log1p(np.maximum(y_pred, 0))
    rmsle = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    nrmse_mean = rmse / np.mean(y_true)
    nrmse_range = rmse / (np.max(y_true) - np.min(y_true))

    print("\nMetric diagnostics")
    print("------------------")
    print("RMSE:", rmse)
    print("MAE:", mae)
    print("RMSLE:", rmsle)
    print("NRMSE (mean):", nrmse_mean)
    print("NRMSE (range):", nrmse_range)


# -----------------------------
# Experiment name
# -----------------------------
experiment_name = "exp22_elasticnet_strong_regularization_20260326"


# -----------------------------
# Load data
# -----------------------------
def load_data():

    print("Loading data...")

    train = pd.read_csv("../data/train.csv", encoding="cp932")
    test = pd.read_csv("../data/test.csv", encoding="cp932")

    print("Train shape:", train.shape)
    print("Test shape:", test.shape)

    return train, test


# -----------------------------
# Prepare features
# -----------------------------
def prepare_features(train, test):

    spectral_cols = [
        c for c in train.columns
        if c not in ["sample number", "species number", "樹種", "含水率"]
    ]

    X = train[spectral_cols].values
    y = train["含水率"].values
    X_test = test[spectral_cols].values

    print("Number of spectral features:", X.shape[1])

    return X, y, X_test


# -----------------------------
# Scale features
# -----------------------------
def scale_features(X, X_test):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(X)
    X_test_scaled = scaler.transform(X_test)

    print("Feature scaling complete")

    return X_scaled, X_test_scaled


# -----------------------------
# Cross validation
# -----------------------------
def cross_validate(X, y):

    print("\nRunning 5-fold cross-validation...")

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    rmse_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        print(f"\nTraining fold {fold+1}")

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        model = ElasticNet(
            alpha=0.001,
            l1_ratio=0.95,
            max_iter=20000,
            tol=1e-3,
            selection="random",
            random_state=42
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmse_scores.append(rmse)

        print("Fold RMSE:", rmse)

        metric_diagnostics(y_val, preds)

    print("\nMean CV RMSE:", np.mean(rmse_scores))


# -----------------------------
# Ensemble prediction
# -----------------------------
def kfold_ensemble(X, y, X_test):

    print("\nTraining KFold ElasticNet ensemble...")

    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    test_preds = np.zeros(X_test.shape[0])

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

        print(f"Training ensemble model {fold+1}")

        X_train = X[train_idx]
        y_train = y[train_idx]

        model = ElasticNet(
            alpha=0.001,
            l1_ratio=0.95,
            max_iter=20000,
            tol=1e-3,
            selection="random",
            random_state=42
        )

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        test_preds += preds / 5

    print("Sample ensemble predictions:", test_preds[:10])

    return test_preds


# -----------------------------
# Save submission
# -----------------------------
def save_submission(test, preds):

    os.makedirs("../submissions", exist_ok=True)

    submission = pd.DataFrame({
        "sample number": test["sample number"],
        "含水率": preds
    })

    output_path = f"../submissions/{experiment_name}.csv"

    submission.to_csv(output_path, index=False, header=False)

    print("\nSubmission saved to:", output_path)

    check = pd.read_csv(output_path, header=None)
    print(check.head())


# -----------------------------
# Main
# -----------------------------
def main():

    train, test = load_data()

    X, y, X_test = prepare_features(train, test)

    X_scaled, X_test_scaled = scale_features(X, X_test)

    cross_validate(X_scaled, y)

    preds = kfold_ensemble(X_scaled, y, X_test_scaled)

    save_submission(test, preds)


main()

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Number of spectral features: 1555
Feature scaling complete

Running 5-fold cross-validation...

Training fold 1
Fold RMSE: 11.886261600271814

Metric diagnostics
------------------
RMSE: 11.886261600271814
MAE: 8.072617094465546
RMSLE: 0.34898912310444175
NRMSE (mean): 0.23815282454128658
NRMSE (range): 0.042513100647468494

Training fold 2
Fold RMSE: 12.518377075055888

Metric diagnostics
------------------
RMSE: 12.518377075055888
MAE: 7.727198680734866
RMSLE: 0.4026282962030683
NRMSE (mean): 0.23824377693089882
NRMSE (range): 0.043558547311947415

Training fold 3
Fold RMSE: 13.155901897382945

Metric diagnostics
------------------
RMSE: 13.155901897382945
MAE: 8.748129134833569
RMSLE: 0.42443319596035267
NRMSE (mean): 0.2746832079461764
NRMSE (range): 0.05559980306164774

Training fold 4
Fold RMSE: 9.932115227755544

Metric diagnostics
------------------
RMSE: 9.932115227755544
MAE: 6.7545310048564176
RMSLE: 0.4080253